# Unified Evaluation Specification & Benchmark Harness
This notebook provides a production-ready, top-to-bottom executable harness for evaluating fretboard position decoders and note repair strategies on **GuitarSet** (in-distribution) and **GAPS** (out-of-distribution) datasets under identical conditions.



In [1]:
# -----------------------------------------------------------------------------
# Imports and path configuration
# -----------------------------------------------------------------------------
import sys
import gzip
import json
import time
import math
import warnings
import copy
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut
import xgboost as xgb
from scipy.optimize import linear_sum_assignment

# Add repo root and backend to sys.path dynamically
cwd = Path.cwd()
repo_root = None
for parent in [cwd] + list(cwd.parents):
    if (parent / ".git").exists() or (parent / "package.json").exists():
        repo_root = parent
        break
if repo_root is None:
    repo_root = cwd

sys.path.append(str(repo_root))
sys.path.append(str(repo_root / "backend"))

# Import production-grade modules
import backend.fretboard as fb
from backend.jams_processor import process_jams_file
from backend.transcribe import snap_notes_to_key
from recode import filter_phantom_notes

# -----------------------------------------------------------------------------
# Global configurations & seed settings
# -----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_GAPS = True
SMOKE_TEST = False           # Set to False to run the full evaluation suite
MAX_EVAL_TRACKS = 8 if SMOKE_TEST else 360
BEAM_WIDTH = 8
ONSET_TOL = 0.035
PRECISION_GATE = 0.85
BP_MODE = "cache_only"
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Device: {DEVICE}")


Device: cuda


## 1. Artifact Verification & DadaGP Transition Prior Compilation


In [2]:
# -----------------------------------------------------------------------------
# Define data paths for artifacts and verify their existence
# -----------------------------------------------------------------------------
GUITARSET_JAMS_DIR = repo_root / "FullGuitarSetData" / "JamsFiles"
BP_CACHE_DIR = repo_root / "outputs" / "audio_to_tab_basic_pitch_heldout" / "basic_pitch_note_cache"
DADAGP_DIR = repo_root / "dadagp_distilled"
GAPS_DIR = repo_root / "GAPS_dataset"
TRANSFORMER_WEIGHTS_PATH = repo_root / "backend" / "tab_transformer_final.pt"

# Verify transformer weights path fallback
if not TRANSFORMER_WEIGHTS_PATH.exists():
    TRANSFORMER_WEIGHTS_PATH = repo_root / "tab_transformer_final.pt"

if not TRANSFORMER_WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Transformer weights file not found at {TRANSFORMER_WEIGHTS_PATH}")
print("Transformer weights exist:", TRANSFORMER_WEIGHTS_PATH.exists())

# -----------------------------------------------------------------------------
# Parse distilled token shards from DadaGP to compute transition priors
# -----------------------------------------------------------------------------
dadagp_priors = defaultdict(float)
total_bigrams = 1.0

try:
    shard_files = sorted(list(DADAGP_DIR.glob("shard_*.jsonl.gz")))
    if shard_files:
        print(f"Loading transitions from ALL {len(shard_files)} DadaGP distilled shards...")
        OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
        for shard_file in shard_files:
            with gzip.open(shard_file, "rt") as f:
                for line in f:
                    data = json.loads(line)
                    notes = []
                    # Accumulate note MIDI pitches using standard offsets
                    for item in data.get("events", []):
                        if item[0] == "g":
                            for s, f_val in item[1]:
                                notes.append(OPEN_STRING_MIDI[s] + f_val)
                    # Count consecutive note transitions
                    for i in range(len(notes) - 1):
                        bigram = (notes[i], notes[i+1])
                        dadagp_priors[bigram] += 1.0
                        total_bigrams += 1.0
        print(f"Loaded {len(dadagp_priors)} transitions from DadaGP. Total bigrams counted: {total_bigrams}")
    else:
        print("No DadaGP distilled shards found. Using flat uniform prior fallback.")
except Exception as e:
    print("DadaGP prior compilation error:", e)

# -----------------------------------------------------------------------------
# Function to get bigram prior probability
# -----------------------------------------------------------------------------
def get_transition_prior(m1, m2):
    val = dadagp_priors.get((m1, m2), 0.0)
    return max(1e-5, (val + 1.0) / (total_bigrams + 128))  # Laplace-smoothed prior


Transformer weights exist: True
Loading transitions from ALL 34 DadaGP distilled shards...
Loaded 2353 transitions from DadaGP. Total bigrams counted: 41732075.0


## 2. Models & Feature Engineering Primitives (8-Feature Set)


In [3]:
# -----------------------------------------------------------------------------
# PyTorch Sequence Transformer Classifier architecture definition
# -----------------------------------------------------------------------------
class PreDecoderTransformer(nn.Module):
    def __init__(self, input_dim, d_model=16, nhead=2, num_layers=2, num_classes=3):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=32, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, num_classes)
        
    def forward(self, x):
        x_proj = self.proj(x)
        out = self.transformer(x_proj)
        logits = self.fc(out[:, -1, :])
        return logits

# -----------------------------------------------------------------------------
# Compile sliding windows from sequence history for Transformer model
# -----------------------------------------------------------------------------
def make_sequences_grouped(X_data, y_data, groups_data, seq_len=5):
    seq_x, seq_y, seq_groups = [], [], []
    unique_groups = np.unique(groups_data)
    for g in unique_groups:
        g_mask = (groups_data == g)
        g_X = X_data[g_mask]
        g_y = y_data[g_mask]
        for j in range(seq_len, len(g_X)):
            seq_x.append(g_X[j-seq_len:j])
            seq_y.append(g_y[j-1])
            seq_groups.append(g)
    return np.array(seq_x), np.array(seq_y), np.array(seq_groups)

def make_eval_sequences(X_track, seq_len=5):
    X_seq_list = []
    for i in range(len(X_track)):
        if i < seq_len - 1:
            pad_size = (seq_len - 1) - i
            pad_feats = [X_track[0]] * pad_size + list(X_track[:i+1])
            X_seq_list.append(pad_feats)
        else:
            X_seq_list.append(X_track[i-seq_len+1:i+1])
    return np.array(X_seq_list)

# -----------------------------------------------------------------------------
# Parse ground-truth MIDI parameters safely from JAMS files
# -----------------------------------------------------------------------------
def jams_to_notes(jams_path):
    rec = process_jams_file(jams_path)
    notes = []
    for n in rec.get("notes", []):
        midi = int(round(float(n["midi"])))
        notes.append({
            "start": float(n["start"]),
            "duration": float(n["duration"]),
            "midi": midi,
            "pitch_class": midi % 12,
            "amplitude": float(n.get("amplitude", 1.0)),
            "true_string": int(n.get("true_string", 0)),
            "true_fret": int(n.get("true_fret", 0))
        })
    notes.sort(key=lambda x: x["start"])
    for idx, tn in enumerate(notes):
        tn["note_idx"] = idx
    return notes


## 3. LOPO Data Collection & Alignment (Labeling Verification)


In [4]:
# -----------------------------------------------------------------------------
# Pair GT files with corresponding Basic Pitch CSVs from cache folder
# -----------------------------------------------------------------------------
recording_datasets = {}
jams_files = sorted(list(GUITARSET_JAMS_DIR.glob("*.jams")))
for jf in jams_files:
    stem = jf.stem
    csv_files = list(BP_CACHE_DIR.glob(f"{stem}*.csv"))
    if csv_files:
        recording_datasets[stem] = (jf, csv_files[0])

print(f"Scanning and aligning {len(recording_datasets)} matched recordings...")

# -----------------------------------------------------------------------------
# Compile player groups and feature sets across GuitarSet tracks
# -----------------------------------------------------------------------------
all_X = []
all_y = []
player_groups = []

for stem, (jams_path, bp_path) in recording_datasets.items():
    player_id = stem.split("_")[0] if "_" in stem else "unknown"
    gt_notes = jams_to_notes(jams_path)
    for idx, gn in enumerate(gt_notes):
        gn["note_idx"] = idx
        
    bp_df = pd.read_csv(bp_path)
    
    bp_notes = []
    for idx, row in bp_df.iterrows():
        bp_notes.append({
            "start": float(row["start"]),
            "duration": float(row["duration"]),
            "midi": int(round(float(row["midi"]))),
            "pitch_class": int(round(float(row["midi"]))) % 12,
            "amplitude": float(row["amplitude"]),
            "note_idx": idx
        })
        
    if not bp_notes or not gt_notes:
        continue
        
    # Set up key scale structures for feature extraction
    detected_key = fb.detect_key(bp_notes) if hasattr(fb, 'detect_key') else 0
    key_scale = [(detected_key + interval) % 12 for interval in [0, 2, 4, 5, 7, 9, 11]]
    amplitudes = [n["amplitude"] for n in bp_notes]
    sorted_amps = sorted(amplitudes)
    
    # -----------------------------------------------------------------------------
    # Execute Hungarian 1:1 matching for label construction (onset tolerance 35ms)
    # -----------------------------------------------------------------------------
    cost_matrix = np.full((len(bp_notes), len(gt_notes)), 1e6)
    for i_bp, tn in enumerate(bp_notes):
        for j_gt, gn in enumerate(gt_notes):
            time_diff = abs(tn["start"] - gn["start"])
            if time_diff <= ONSET_TOL:
                pitch_diff = abs(tn["midi"] - gn["midi"])
                cost_matrix[i_bp, j_gt] = time_diff * 100.0 + pitch_diff
                
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    bp_to_gt = {}
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] < 1e5:
            bp_to_gt[bp_notes[r]["note_idx"]] = gt_notes[c]
            
    # -----------------------------------------------------------------------------
    # Build 8-feature representation for each note event to prevent leakage
    # -----------------------------------------------------------------------------
    for i, tn in enumerate(bp_notes):
        tn["amp_rank"] = sum(1 for a in sorted_amps if a < tn["amplitude"]) / len(amplitudes)
        window_midis = [w["midi"] for j, w in enumerate(bp_notes) if max(0, i-4) <= j < min(len(bp_notes), i+5) and j != i]
        local_median = np.median(window_midis) if window_midis else tn["midi"]
        tn["register_distance"] = abs(tn["midi"] - local_median)
        tn["in_key"] = 1.0 if (tn["pitch_class"] in key_scale) else 0.0
        
        is_overtone = 0.0
        for other in bp_notes:
            if abs(tn["start"] - other["start"]) <= 0.050:
                if tn["midi"] - other["midi"] in [12, 19, 24]:
                    if other["amplitude"] > tn["amplitude"]:
                        is_overtone = 1.0
                        break
        tn["is_overtone"] = is_overtone
        prev_midi = bp_notes[i-1]["midi"] if i > 0 else tn["midi"]
        tn["prior_prob"] = get_transition_prior(prev_midi, tn["midi"])
        tn["ioi"] = tn["start"] - bp_notes[i-1]["start"] if i > 0 else 0.0
        tn["note_density"] = sum(1.0 for other in bp_notes if abs(tn["start"] - other["start"]) <= 0.1)
        
        # -----------------------------------------------------------------------------
        # Correct classification targets: Valid (0), Phantom (1), Octave Slip (2)
        # -----------------------------------------------------------------------------
        if tn["note_idx"] in bp_to_gt:
            best_match = bp_to_gt[tn["note_idx"]]
            if best_match["midi"] == tn["midi"]:
                tn["label"] = 0
            elif abs(best_match["midi"] - tn["midi"]) in [12, 24]:
                tn["label"] = 2
            else:
                tn["label"] = 1
        else:
            tn["label"] = 1
            
        row_feat = [
            tn["amp_rank"], tn["duration"], tn["register_distance"], 
            tn["in_key"], tn["is_overtone"], tn["prior_prob"],
            tn["ioi"], tn["note_density"]
        ]
        all_X.append(row_feat)
        all_y.append(tn["label"])
        player_groups.append(player_id)

X = np.array(all_X)
y = np.array(all_y)
groups = np.array(player_groups)

print(f"Alignment verification: Valid={sum(y==0)} | Phantom={sum(y==1)} | Octave-slip={sum(y==2)}")


Scanning and aligning 54 matched recordings...
Alignment verification: Valid=7253 | Phantom=2243 | Octave-slip=65


## 4. Shared Primitives (Scoring, Downstream/Upstream Repair, and Decoders)


In [5]:
def group_notes_by_onset(notes):
    groups = []
    current_g = []
    for n in sorted(notes, key=lambda x: x["start"]):
        if not current_g:
            current_g.append(n)
        elif abs(n["start"] - current_g[0]["start"]) <= 0.035:
            current_g.append(n)
        else:
            groups.append(current_g)
            current_g = [n]
    if current_g:
        groups.append(current_g)
    return groups

def repair_notebook_tab_assignments(assigned_rows):
    if not assigned_rows:
        return []
    repaired = [dict(r) for r in assigned_rows]
    groups = group_notes_by_onset(repaired)
    OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
    # Get possible positions helper
    def get_possible_positions(midi):
        pos = []
        for s, open_val in enumerate(OPEN_STRING_MIDI):
            fret = midi - open_val
            if 0 <= fret <= 24:
                pos.append({"string": s, "fret": fret})
        return pos
        
    for g in groups:
        strings = [n.get("pred_string") for n in g]
        if len(strings) != len(set(strings)):
            used_strings = set()
            for n in g:
                midi = n["midi"]
                poss = get_possible_positions(midi)
                avail = [p for p in poss if p["string"] not in used_strings]
                if avail:
                    best = min(avail, key=lambda x: x["fret"])
                    n["pred_string"] = best["string"]
                    n["pred_fret"] = best["fret"]
                    used_strings.add(best["string"])
                else:
                    used_strings.add(n["pred_string"])
    return repaired

# -----------------------------------------------------------------------------
# Exact Tab Accuracy Position Scoring logic (via stable note_idx mapping)
# -----------------------------------------------------------------------------
def score_exact_tab(decoded, bp_to_gt):
    exact = 0
    total = 0
    for tn_dec in decoded:
        note_idx = tn_dec.get("note_idx")
        if note_idx is not None and note_idx in bp_to_gt:
            orig_match = bp_to_gt[note_idx]
            total += 1
            if int(tn_dec["pred_string"]) == int(orig_match["true_string"]) and int(tn_dec["pred_fret"]) == int(orig_match["true_fret"]):
                exact += 1
    return exact / total if total > 0 else 0.0

# -----------------------------------------------------------------------------
# Audit physical violations: string collisions and CAGED stretch bounds
# -----------------------------------------------------------------------------
def audit_playability(decoded_notes):
    collisions = 0
    stretch_violations = 0
    groups = group_notes_by_onset(decoded_notes)
    for g in groups:
        strings = [n.get("pred_string") for n in g if n.get("pred_string") is not None]
        if len(strings) != len(set(strings)):
            collisions += (len(strings) - len(set(strings)))
        frets = [n.get("pred_fret") for n in g if n.get("pred_fret") is not None and n.get("pred_fret") > 0]
        if frets:
            span = max(frets) - min(frets)
            if span > 4:
                stretch_violations += 1
    return {"collisions": collisions, "stretch_violations": stretch_violations}

# -----------------------------------------------------------------------------
# 2D Threshold Cascade sweeping optimization logic
# -----------------------------------------------------------------------------
def apply_cascading_classifier(probs, valid_threshold, error_threshold=0.5):
    preds = np.zeros(len(probs), dtype=int)
    for idx, p in enumerate(probs):
        if p[0] >= valid_threshold:
            preds[idx] = 0
        else:
            sum_err = p[1] + p[2]
            if sum_err > 0:
                p2_norm = p[2] / sum_err
                preds[idx] = 2 if p2_norm >= error_threshold else 1
            else:
                preds[idx] = 1
    return preds

def sweep_threshold_2d(probs, y_true):
    best_t_valid = 0.5
    best_t_error = 0.5
    best_f1 = -1.0
    best_p = 0.0
    precision_gate_passed = False
    
    for t_val in np.linspace(0.05, 0.95, 19):
        for t_err in np.linspace(0.1, 0.9, 9):
            preds = apply_cascading_classifier(probs, t_val, t_err)
            p = precision_score(y_true, preds, average='macro', zero_division=0)
            f = f1_score(y_true, preds, average='macro', zero_division=0)
            if p >= PRECISION_GATE:
                if not precision_gate_passed:
                    best_f1 = -1.0  # Reset F1 to prioritize gate-passing thresholds
                precision_gate_passed = True
                if f > best_f1:
                    best_f1 = f
                    best_t_valid = t_val
                    best_t_error = t_err
                    best_p = p
            else:
                if not precision_gate_passed and p > best_p:
                    best_p = p
                    best_t_valid = t_val
                    best_t_error = t_err
                    best_f1 = f
    return best_t_valid, best_t_error, precision_gate_passed


## 5. Model Cross-Validation (LOPO Loop & Threshold Parameters)


In [6]:
# -----------------------------------------------------------------------------
# Configure training parameters for Exploration (v1) and E2E (v2) models
# -----------------------------------------------------------------------------
v1_params = {
    "rf": {"n_estimators": 100, "max_depth": 10, "class_weight": "balanced", "random_state": SEED},
    "xgb": {"n_estimators": 100, "max_depth": 5, "learning_rate": 0.1, "random_state": SEED, "eval_metric": "mlogloss", "objective": "multi:softprob"},
    "mlp": {"hidden_layer_sizes": (64, 32), "max_iter": 200, "random_state": SEED}
}

v2_params = {
    "rf": {"n_estimators": 200, "max_depth": 12, "class_weight": "balanced", "random_state": SEED},
    "xgb": {"n_estimators": 150, "max_depth": 6, "learning_rate": 0.05, "random_state": SEED, "eval_metric": "mlogloss", "objective": "multi:softprob"},
    "mlp": {"hidden_layer_sizes": (32, 16), "max_iter": 300, "random_state": SEED}
}

logo = LeaveOneGroupOut()
folds_data = {}

def train_seq_model(X_train_seq, y_train, epochs=5):
    model = PreDecoderTransformer(input_dim=X_train_seq.shape[2], num_classes=3).to(DEVICE)
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=0.001)
    ds = TensorDataset(torch.tensor(X_train_seq, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    dl = DataLoader(ds, batch_size=128, shuffle=True)
    model.train()
    for _ in range(epochs):
        for bx, by in dl:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(bx), by)
            loss.backward()
            opt.step()
    return model

def predict_seq_probs(model, X_val_seq):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_val_seq, dtype=torch.float32).to(DEVICE)
        out = model(X_tensor)
        return torch.softmax(out, dim=1).cpu().numpy()

# -----------------------------------------------------------------------------
# Train loop iterating across player validation splits
# -----------------------------------------------------------------------------
print("Training cross-validation models...")
for fold, (train_idx, val_idx) in enumerate(logo.split(X, y, groups)):
    X_train_raw, X_val_raw = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    groups_train, groups_val = groups[train_idx], groups[val_idx]
    val_group = groups_val[0]
    
    # Local standardization to prevent data leakage
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    
    oof_probs_xgb_v1 = np.zeros((len(y_train), 3))
    oof_probs_xgb_v2 = np.zeros((len(y_train), 3))
    oof_probs_rf_v1 = np.zeros((len(y_train), 3))
    oof_probs_rf_v2 = np.zeros((len(y_train), 3))
    oof_probs_mlp_v1 = np.zeros((len(y_train), 3))
    oof_probs_mlp_v2 = np.zeros((len(y_train), 3))
    oof_probs_seq_v1 = np.zeros((len(y_train), 3))
    oof_probs_seq_v2 = np.zeros((len(y_train), 3))
    
    # -----------------------------------------------------------------------------
    # Nested cross-validation for hyperparameter threshold search
    # -----------------------------------------------------------------------------
    nested_logo = LeaveOneGroupOut()
    for n_tr, n_val in nested_logo.split(X_train, y_train, groups_train):
        n_scaler = StandardScaler()
        nX_tr = n_scaler.fit_transform(X_train_raw[n_tr])
        nX_val = n_scaler.transform(X_train_raw[n_val])
        ny_tr, ny_val = y_train[n_tr], y_train[n_val]
        
        # XGB
        n_xgb_v1 = xgb.XGBClassifier(**v1_params["xgb"]).fit(nX_tr, ny_tr)
        oof_probs_xgb_v1[n_val] = n_xgb_v1.predict_proba(nX_val)
        
        n_xgb_v2 = xgb.XGBClassifier(**v2_params["xgb"]).fit(nX_tr, ny_tr)
        oof_probs_xgb_v2[n_val] = n_xgb_v2.predict_proba(nX_val)
        
        # RF
        n_rf_v1 = RandomForestClassifier(**v1_params["rf"]).fit(nX_tr, ny_tr)
        oof_probs_rf_v1[n_val] = n_rf_v1.predict_proba(nX_val)
        
        n_rf_v2 = RandomForestClassifier(**v2_params["rf"]).fit(nX_tr, ny_tr)
        oof_probs_rf_v2[n_val] = n_rf_v2.predict_proba(nX_val)
        
        # MLP
        n_mlp_v1 = MLPClassifier(**v1_params["mlp"]).fit(nX_tr, ny_tr)
        oof_probs_mlp_v1[n_val] = n_mlp_v1.predict_proba(nX_val)
        
        n_mlp_v2 = MLPClassifier(**v2_params["mlp"]).fit(nX_tr, ny_tr)
        oof_probs_mlp_v2[n_val] = n_mlp_v2.predict_proba(nX_val)
        
        # Seq Transformers (Inner-fold sequence conversion)
        ngroups_tr = groups_train[n_tr]
        nX_tr_seq = np.zeros((len(nX_tr), 5, nX_tr.shape[1]))
        for g in np.unique(ngroups_tr):
            g_mask = (ngroups_tr == g)
            nX_tr_seq[g_mask] = make_eval_sequences(nX_tr[g_mask])
        nX_val_seq = make_eval_sequences(nX_val)
        
        n_seq_v1 = train_seq_model(nX_tr_seq, ny_tr, epochs=5)
        oof_probs_seq_v1[n_val] = predict_seq_probs(n_seq_v1, nX_val_seq)
        
        n_seq_v2 = train_seq_model(nX_tr_seq, ny_tr, epochs=8)
        oof_probs_seq_v2[n_val] = predict_seq_probs(n_seq_v2, nX_val_seq)
        
    # Sweep thresholds
    t_v_xgb_v1, t_e_xgb_v1, gate_xgb_v1 = sweep_threshold_2d(oof_probs_xgb_v1, y_train)
    t_v_xgb_v2, t_e_xgb_v2, gate_xgb_v2 = sweep_threshold_2d(oof_probs_xgb_v2, y_train)
    t_v_rf_v1, t_e_rf_v1, gate_rf_v1 = sweep_threshold_2d(oof_probs_rf_v1, y_train)
    t_v_rf_v2, t_e_rf_v2, gate_rf_v2 = sweep_threshold_2d(oof_probs_rf_v2, y_train)
    t_v_mlp_v1, t_e_mlp_v1, gate_mlp_v1 = sweep_threshold_2d(oof_probs_mlp_v1, y_train)
    t_v_mlp_v2, t_e_mlp_v2, gate_mlp_v2 = sweep_threshold_2d(oof_probs_mlp_v2, y_train)
    t_v_seq_v1, t_e_seq_v1, gate_seq_v1 = sweep_threshold_2d(oof_probs_seq_v1, y_train)
    t_v_seq_v2, t_e_seq_v2, gate_seq_v2 = sweep_threshold_2d(oof_probs_seq_v2, y_train)
    
    xgb_v1 = xgb.XGBClassifier(**v1_params["xgb"]).fit(X_train, y_train)
    xgb_v2 = xgb.XGBClassifier(**v2_params["xgb"]).fit(X_train, y_train)
    rf_v1 = RandomForestClassifier(**v1_params["rf"]).fit(X_train, y_train)
    rf_v2 = RandomForestClassifier(**v2_params["rf"]).fit(X_train, y_train)
    mlp_v1 = MLPClassifier(**v1_params["mlp"]).fit(X_train, y_train)
    mlp_v2 = MLPClassifier(**v2_params["mlp"]).fit(X_train, y_train)
    
    # Outer sequence conversion and model training (completely decoupled)
    X_train_seq_all = np.zeros((len(X_train), 5, X_train.shape[1]))
    for g in np.unique(groups_train):
        g_mask = (groups_train == g)
        X_train_seq_all[g_mask] = make_eval_sequences(X_train[g_mask])
        
    seq_trans_v1 = train_seq_model(X_train_seq_all, y_train, epochs=10)
    seq_trans_v2 = train_seq_model(X_train_seq_all, y_train, epochs=20)
    
    # Evaluate on the validation set for local fold-level metrics
    val_preds = {}
    val_preds["xgb_v1"] = apply_cascading_classifier(xgb_v1.predict_proba(X_val), t_v_xgb_v1, t_e_xgb_v1)
    val_preds["xgb_v2"] = apply_cascading_classifier(xgb_v2.predict_proba(X_val), t_v_xgb_v2, t_e_xgb_v2)
    val_preds["rf_v1"] = apply_cascading_classifier(rf_v1.predict_proba(X_val), t_v_rf_v1, t_e_rf_v1)
    val_preds["rf_v2"] = apply_cascading_classifier(rf_v2.predict_proba(X_val), t_v_rf_v2, t_e_rf_v2)
    val_preds["mlp_v1"] = apply_cascading_classifier(mlp_v1.predict_proba(X_val), t_v_mlp_v1, t_e_mlp_v1)
    val_preds["mlp_v2"] = apply_cascading_classifier(mlp_v2.predict_proba(X_val), t_v_mlp_v2, t_e_mlp_v2)
    
    X_val_seq = make_eval_sequences(X_val)
    probs_seq_v1 = predict_seq_probs(seq_trans_v1, X_val_seq)
    probs_seq_v2 = predict_seq_probs(seq_trans_v2, X_val_seq)
    val_preds["seq_v1"] = apply_cascading_classifier(probs_seq_v1, t_v_seq_v1, t_e_seq_v1)
    val_preds["seq_v2"] = apply_cascading_classifier(probs_seq_v2, t_v_seq_v2, t_e_seq_v2)
    
    fold_metrics = {}
    for name, prs in val_preds.items():
        p = precision_score(y_val, prs, average='macro', zero_division=0)
        r = recall_score(y_val, prs, average='macro', zero_division=0)
        f = f1_score(y_val, prs, average='macro', zero_division=0)
        fold_metrics[name] = {"precision": float(p), "recall": float(r), "f1": float(f)}
        
    # Store training metadata per group
    folds_data[val_group] = {
        "scaler": scaler,
        "xgb_v1": xgb_v1, "xgb_v2": xgb_v2,
        "rf_v1": rf_v1, "rf_v2": rf_v2,
        "mlp_v1": mlp_v1, "mlp_v2": mlp_v2,
        "seq_trans_v1": seq_trans_v1, "seq_trans_v2": seq_trans_v2,
        "thresholds": {
            "xgb_v1": (t_v_xgb_v1, t_e_xgb_v1),
            "xgb_v2": (t_v_xgb_v2, t_e_xgb_v2),
            "rf_v1": (t_v_rf_v1, t_e_rf_v1),
            "rf_v2": (t_v_rf_v2, t_e_rf_v2),
            "mlp_v1": (t_v_mlp_v1, t_e_mlp_v1),
            "mlp_v2": (t_v_mlp_v2, t_e_mlp_v2),
            "seq_v1": (t_v_seq_v1, t_e_seq_v1),
            "seq_v2": (t_v_seq_v2, t_e_seq_v2)
        },
        "gates": {
            "xgb_v1": gate_xgb_v1, "xgb_v2": gate_xgb_v2,
            "rf_v1": gate_rf_v1, "rf_v2": gate_rf_v2,
            "mlp_v1": gate_mlp_v1, "mlp_v2": gate_mlp_v2,
            "seq_v1": gate_seq_v1, "seq_v2": gate_seq_v2
        },
        "val_metrics": fold_metrics
    }
print("LOPO Cross-Validation training completed.")


Training cross-validation models...
LOPO Cross-Validation training completed.


## 6. Mode A & Mode B Dual-Protocol Benchmark Runner


In [7]:
# -----------------------------------------------------------------------------
# Adjudicator repair mapping logic definition
# -----------------------------------------------------------------------------
def repair_notes(notes, predictions):
    repaired = []
    for i, tn in enumerate(notes):
        pred = predictions[i]
        if pred == 0:
            repaired.append(dict(tn))
        elif pred == 2:
            window_m = [w["midi"] for j, w in enumerate(notes) if max(0, i-4) <= j < min(len(notes), i+5) and j != i]
            local_med = np.median(window_m) if window_m else tn["midi"]
            direction = -12 if tn["midi"] > local_med else 12
            rep_note = dict(tn)
            rep_note["midi"] = int(tn["midi"] + direction)
            rep_note["pitch_class"] = rep_note["midi"] % 12
            repaired.append(rep_note)
    return repaired

# Metrics containers initialization
mode_a_results = defaultdict(list)
mode_b_results = defaultdict(list)
mode_a_collisions = defaultdict(list)
mode_a_stretches = defaultdict(list)
mode_a_latencies = defaultdict(list)

eval_count = 0
print("Running E2E and Oracle evaluations across GuitarSet stems...")

# ------------------------------------------------
# Main validation run looping over test recordings
# ------------------------------------------------
for stem, (jams_path, bp_path) in recording_datasets.items():
    if eval_count >= MAX_EVAL_TRACKS:
        break
        
    player_id = stem.split("_")[0] if "_" in stem else "unknown"
    if player_id not in folds_data:
        continue
        
    eval_count += 1
    fold_info = folds_data[player_id]
    scaler = fold_info["scaler"]
    
    gt_notes = jams_to_notes(jams_path)
    for idx, gn in enumerate(gt_notes):
        gn["note_idx"] = idx
        
    bp_df = pd.read_csv(bp_path)
    bp_notes = []
    for idx, row in bp_df.iterrows():
        bp_notes.append({
            "start": float(row["start"]),
            "duration": float(row["duration"]),
            "midi": int(round(float(row["midi"]))),
            "pitch_class": int(round(float(row["midi"]))) % 12,
            "amplitude": float(row["amplitude"]),
            "note_idx": idx
        })
        
    # Standard Alignment Map construction
    cost_matrix = np.full((len(bp_notes), len(gt_notes)), 1e6)
    for i_bp, tn in enumerate(bp_notes):
        for j_gt, gn in enumerate(gt_notes):
            time_diff = abs(tn["start"] - gn["start"])
            if time_diff <= ONSET_TOL:
                pitch_diff = abs(tn["midi"] - gn["midi"])
                cost_matrix[i_bp, j_gt] = time_diff * 100.0 + pitch_diff
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    bp_to_gt = {}
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] < 1e5:
            bp_to_gt[bp_notes[r]["note_idx"]] = gt_notes[c]
            
    # Compile track features
    amplitudes = [n["amplitude"] for n in bp_notes]
    sorted_amps = sorted(amplitudes)
    detected_key = fb.detect_key(bp_notes) if hasattr(fb, 'detect_key') else 0
    key_scale = [(detected_key + interval) % 12 for interval in [0, 2, 4, 5, 7, 9, 11]]
    track_features = []
    for i, tn in enumerate(bp_notes):
        amp_rank = sum(1 for a in sorted_amps if a < tn["amplitude"]) / len(amplitudes)
        window_midis = [w["midi"] for j, w in enumerate(bp_notes) if max(0, i-4) <= j < min(len(bp_notes), i+5) and j != i]
        local_median = np.median(window_midis) if window_midis else tn["midi"]
        reg_dist = abs(tn["midi"] - local_median)
        in_key = 1.0 if (tn["pitch_class"] in key_scale) else 0.0
        
        is_overtone = 0.0
        for other in bp_notes:
            if abs(tn["start"] - other["start"]) <= 0.050:
                if tn["midi"] - other["midi"] in [12, 19, 24]:
                    if other["amplitude"] > tn["amplitude"]:
                        is_overtone = 1.0
                        break
        prev_midi = bp_notes[i-1]["midi"] if i > 0 else tn["midi"]
        prior_prob = get_transition_prior(prev_midi, tn["midi"])
        ioi = tn["start"] - bp_notes[i-1]["start"] if i > 0 else 0.0
        note_density = sum(1.0 for other in bp_notes if abs(tn["start"] - other["start"]) <= 0.1)
        
        track_features.append([amp_rank, tn["duration"], reg_dist, in_key, is_overtone, prior_prob, ioi, note_density])
        
    X_track = scaler.transform(np.array(track_features))
    
    # Predict for each classifier model (v1 vs v2)
    probs_xgb_v1 = fold_info["xgb_v1"].predict_proba(X_track)
    probs_xgb_v2 = fold_info["xgb_v2"].predict_proba(X_track)
    probs_rf_v1 = fold_info["rf_v1"].predict_proba(X_track)
    probs_rf_v2 = fold_info["rf_v2"].predict_proba(X_track)
    probs_mlp_v1 = fold_info["mlp_v1"].predict_proba(X_track)
    probs_mlp_v2 = fold_info["mlp_v2"].predict_proba(X_track)
    
    seq_trans_v1 = fold_info["seq_trans_v1"]
    seq_trans_v2 = fold_info["seq_trans_v2"]
    
    # Sequence formatting for sequence transformer
    X_seq_list = make_eval_sequences(X_track)
    X_seq_tensor = torch.tensor(X_seq_list, dtype=torch.float32).to(DEVICE)
    
    seq_trans_v1.eval()
    seq_trans_v2.eval()
    with torch.no_grad():
        out_v1 = seq_trans_v1(X_seq_tensor)
        probs_seq_v1 = torch.softmax(out_v1, dim=1).cpu().numpy()
        
        out_v2 = seq_trans_v2(X_seq_tensor)
        probs_seq_v2 = torch.softmax(out_v2, dim=1).cpu().numpy()
        
    # Get sweeps validation thresholds
    t_v_xgb_v1, t_e_xgb_v1 = fold_info["thresholds"]["xgb_v1"]
    t_v_xgb_v2, t_e_xgb_v2 = fold_info["thresholds"]["xgb_v2"]
    t_v_rf_v1, t_e_rf_v1 = fold_info["thresholds"]["rf_v1"]
    t_v_rf_v2, t_e_rf_v2 = fold_info["thresholds"]["rf_v2"]
    t_v_mlp_v1, t_e_mlp_v1 = fold_info["thresholds"]["mlp_v1"]
    t_v_mlp_v2, t_e_mlp_v2 = fold_info["thresholds"]["mlp_v2"]
    t_v_seq_v1, t_e_seq_v1 = fold_info["thresholds"]["seq_v1"]
    t_v_seq_v2, t_e_seq_v2 = fold_info["thresholds"]["seq_v2"]
    
    # Evaluate predicted actions
    preds_xgb_v1 = apply_cascading_classifier(probs_xgb_v1, t_v_xgb_v1, t_e_xgb_v1)
    preds_xgb_v2 = apply_cascading_classifier(probs_xgb_v2, t_v_xgb_v2, t_e_xgb_v2)
    preds_rf_v1 = apply_cascading_classifier(probs_rf_v1, t_v_rf_v1, t_e_rf_v1)
    preds_rf_v2 = apply_cascading_classifier(probs_rf_v2, t_v_rf_v2, t_e_rf_v2)
    preds_mlp_v1 = apply_cascading_classifier(probs_mlp_v1, t_v_mlp_v1, t_e_mlp_v1)
    preds_mlp_v2 = apply_cascading_classifier(probs_mlp_v2, t_v_mlp_v2, t_e_mlp_v2)
    preds_seq_v1 = apply_cascading_classifier(probs_seq_v1, t_v_seq_v1, t_e_seq_v1)
    preds_seq_v2 = apply_cascading_classifier(probs_seq_v2, t_v_seq_v2, t_e_seq_v2)
    
    # Reconstructed repaired note lists
    rep_xgb_v1 = repair_notes(bp_notes, preds_xgb_v1)
    rep_xgb_v2 = repair_notes(bp_notes, preds_xgb_v2)
    rep_rf_v1 = repair_notes(bp_notes, preds_rf_v1)
    rep_rf_v2 = repair_notes(bp_notes, preds_rf_v2)
    rep_mlp_v1 = repair_notes(bp_notes, preds_mlp_v1)
    rep_mlp_v2 = repair_notes(bp_notes, preds_mlp_v2)
    rep_seq_v1 = repair_notes(bp_notes, preds_seq_v1)
    rep_seq_v2 = repair_notes(bp_notes, preds_seq_v2)
    
    # ------------------
    # MODE A: Transcription E2E
    # ------------------
    t0 = time.time()
    dec_base_a = fb.assign_combined_all_tuned(copy.deepcopy(bp_notes))
    mode_a_latencies["baseline"].append(time.time() - t0)
    mode_a_results["baseline"].append(score_exact_tab(dec_base_a, bp_to_gt))
    aud = audit_playability(dec_base_a)
    mode_a_collisions["baseline"].append(aud["collisions"])
    mode_a_stretches["baseline"].append(aud["stretch_violations"])
    
    t0 = time.time()
    dec_down_a = repair_notebook_tab_assignments(dec_base_a)
    mode_a_latencies["downstream"].append(time.time() - t0)
    mode_a_results["downstream"].append(score_exact_tab(dec_down_a, bp_to_gt))
    aud = audit_playability(dec_down_a)
    mode_a_collisions["downstream"].append(aud["collisions"])
    mode_a_stretches["downstream"].append(aud["stretch_violations"])
    
    rep_heuristic = filter_phantom_notes(copy.deepcopy(bp_notes), threshold=0.40)
    t0 = time.time()
    dec_heur_vit_a = fb.assign_combined_all_tuned(rep_heuristic)
    mode_a_latencies["heuristic_vit"].append(time.time() - t0)
    mode_a_results["heuristic_vit"].append(score_exact_tab(dec_heur_vit_a, bp_to_gt))
    aud = audit_playability(dec_heur_vit_a)
    mode_a_collisions["heuristic_vit"].append(aud["collisions"])
    mode_a_stretches["heuristic_vit"].append(aud["stretch_violations"])
    
    try:
        t0 = time.time()
        dec_heur_beam_a = fb.assign_prox_viterbi_transformer(rep_heuristic)
        mode_a_latencies["heuristic_beam"].append(time.time() - t0)
        mode_a_results["heuristic_beam"].append(score_exact_tab(dec_heur_beam_a, bp_to_gt))
        aud = audit_playability(dec_heur_beam_a)
        mode_a_collisions["heuristic_beam"].append(aud["collisions"])
        mode_a_stretches["heuristic_beam"].append(aud["stretch_violations"])
    except Exception as e:
        print(f"Heuristic beam Mode A failed on track {stem}: {e}")
        mode_a_results["heuristic_beam"].append(np.nan)
        
    t0 = time.time()
    dec_xgb_v1 = fb.assign_combined_all_tuned(rep_xgb_v1)
    mode_a_latencies["5_v1_xgb"].append(time.time() - t0)
    mode_a_results["5_v1_xgb"].append(score_exact_tab(dec_xgb_v1, bp_to_gt))
    aud = audit_playability(dec_xgb_v1)
    mode_a_collisions["5_v1_xgb"].append(aud["collisions"])
    mode_a_stretches["5_v1_xgb"].append(aud["stretch_violations"])
    
    t0 = time.time()
    dec_rf_v1 = fb.assign_combined_all_tuned(rep_rf_v1)
    mode_a_latencies["5_v1_rf"].append(time.time() - t0)
    mode_a_results["5_v1_rf"].append(score_exact_tab(dec_rf_v1, bp_to_gt))
    aud = audit_playability(dec_rf_v1)
    mode_a_collisions["5_v1_rf"].append(aud["collisions"])
    mode_a_stretches["5_v1_rf"].append(aud["stretch_violations"])
    
    t0 = time.time()
    dec_mlp_v1 = fb.assign_combined_all_tuned(rep_mlp_v1)
    mode_a_latencies["5_v1_mlp"].append(time.time() - t0)
    mode_a_results["5_v1_mlp"].append(score_exact_tab(dec_mlp_v1, bp_to_gt))
    aud = audit_playability(dec_mlp_v1)
    mode_a_collisions["5_v1_mlp"].append(aud["collisions"])
    mode_a_stretches["5_v1_mlp"].append(aud["stretch_violations"])
    
    try:
        t0 = time.time()
        dec_seq_v1 = fb.assign_prox_viterbi_transformer(rep_seq_v1)
        mode_a_latencies["5_v1_seq"].append(time.time() - t0)
        mode_a_results["5_v1_seq"].append(score_exact_tab(dec_seq_v1, bp_to_gt))
        aud = audit_playability(dec_seq_v1)
        mode_a_collisions["5_v1_seq"].append(aud["collisions"])
        mode_a_stretches["5_v1_seq"].append(aud["stretch_violations"])
    except Exception as e:
        print(f"Seq v1 Mode A failed on track {stem}: {e}")
        mode_a_results["5_v1_seq"].append(np.nan)
        
    t0 = time.time()
    dec_xgb_v2 = fb.assign_combined_all_tuned(rep_xgb_v2)
    mode_a_latencies["5_v2_xgb"].append(time.time() - t0)
    mode_a_results["5_v2_xgb"].append(score_exact_tab(dec_xgb_v2, bp_to_gt))
    aud = audit_playability(dec_xgb_v2)
    mode_a_collisions["5_v2_xgb"].append(aud["collisions"])
    mode_a_stretches["5_v2_xgb"].append(aud["stretch_violations"])
    
    t0 = time.time()
    dec_rf_v2 = fb.assign_combined_all_tuned(rep_rf_v2)
    mode_a_latencies["5_v2_rf"].append(time.time() - t0)
    mode_a_results["5_v2_rf"].append(score_exact_tab(dec_rf_v2, bp_to_gt))
    aud = audit_playability(dec_rf_v2)
    mode_a_collisions["5_v2_rf"].append(aud["collisions"])
    mode_a_stretches["5_v2_rf"].append(aud["stretch_violations"])
    
    t0 = time.time()
    dec_mlp_v2 = fb.assign_combined_all_tuned(rep_mlp_v2)
    mode_a_latencies["5_v2_mlp"].append(time.time() - t0)
    mode_a_results["5_v2_mlp"].append(score_exact_tab(dec_mlp_v2, bp_to_gt))
    aud = audit_playability(dec_mlp_v2)
    mode_a_collisions["5_v2_mlp"].append(aud["collisions"])
    mode_a_stretches["5_v2_mlp"].append(aud["stretch_violations"])
    
    try:
        t0 = time.time()
        dec_seq_v2 = fb.assign_prox_viterbi_transformer(rep_seq_v2)
        mode_a_latencies["5_v2_seq"].append(time.time() - t0)
        mode_a_results["5_v2_seq"].append(score_exact_tab(dec_seq_v2, bp_to_gt))
        aud = audit_playability(dec_seq_v2)
        mode_a_collisions["5_v2_seq"].append(aud["collisions"])
        mode_a_stretches["5_v2_seq"].append(aud["stretch_violations"])
    except Exception as e:
        print(f"Seq v2 Mode A failed on track {stem}: {e}")
        mode_a_results["5_v2_seq"].append(np.nan)
        
    # Live path: production transcribe setup (filter_phantom_notes + snap_notes_to_key)
    recoded_live = filter_phantom_notes(copy.deepcopy(bp_notes), threshold=0.40)
    key_info = {"scale_pcs": key_scale}
    snapped_notes = snap_notes_to_key(recoded_live, key_info)
    t0 = time.time()
    dec_live_a = fb.assign_combined_all_tuned(snapped_notes)
    mode_a_latencies["live_backend"].append(time.time() - t0)
    mode_a_results["live_backend"].append(score_exact_tab(dec_live_a, bp_to_gt))
    aud = audit_playability(dec_live_a)
    mode_a_collisions["live_backend"].append(aud["collisions"])
    mode_a_stretches["live_backend"].append(aud["stretch_violations"])
    
    # ------------------
    # MODE B: Oracle Assignment
    # ------------------
    oracle_alignment = {idx: gn for idx, gn in enumerate(gt_notes)}
    
    t0 = time.time()
    dec_base_b = fb.assign_combined_all_tuned(copy.deepcopy(gt_notes))
    mode_b_results["baseline"].append(score_exact_tab(dec_base_b, oracle_alignment))
    
    t0 = time.time()
    dec_down_b = repair_notebook_tab_assignments(dec_base_b)
    mode_b_results["downstream"].append(score_exact_tab(dec_down_b, oracle_alignment))
    
    rep_heur_b = filter_phantom_notes(copy.deepcopy(gt_notes), threshold=0.40)
    t0 = time.time()
    dec_heur_vit_b = fb.assign_combined_all_tuned(rep_heur_b)
    mode_b_results["heuristic_vit"].append(score_exact_tab(dec_heur_vit_b, oracle_alignment))
    
    try:
        dec_heur_beam_b = fb.assign_prox_viterbi_transformer(rep_heur_b)
        mode_b_results["heuristic_beam"].append(score_exact_tab(dec_heur_beam_b, oracle_alignment))
    except Exception:
        mode_b_results["heuristic_beam"].append(np.nan)

# Print summary results to console
print("\nMode A (E2E) Mean Exact Tab Position Accuracies:")
final_accuracies_a = {}
for k, v in mode_a_results.items():
    valid_vals = [x for x in v if not pd.isna(x)]
    mean_val = np.mean(valid_vals) if valid_vals else 0.0
    final_accuracies_a[k] = mean_val
    print(f" - {k}: {mean_val * 100:.2f}%")
    
print("\nMode B (Oracle) Mean Exact Tab Position Accuracies:")
final_accuracies_b = {}
for k, v in mode_b_results.items():
    valid_vals = [x for x in v if not pd.isna(x)]
    mean_val = np.mean(valid_vals) if valid_vals else 0.0
    final_accuracies_b[k] = mean_val
    print(f" - {k}: {mean_val * 100:.2f}%")


Running E2E and Oracle evaluations across GuitarSet stems...

Mode A (E2E) Mean Exact Tab Position Accuracies:
 - baseline: 58.49%
 - downstream: 58.59%
 - heuristic_vit: 60.85%
 - heuristic_beam: 65.29%
 - 5_v1_xgb: 60.54%
 - 5_v1_rf: 59.15%
 - 5_v1_mlp: 59.81%
 - 5_v1_seq: 59.39%
 - 5_v2_xgb: 60.55%
 - 5_v2_rf: 59.02%
 - 5_v2_mlp: 59.79%
 - 5_v2_seq: 61.78%
 - live_backend: 34.89%

Mode B (Oracle) Mean Exact Tab Position Accuracies:
 - baseline: 69.34%
 - downstream: 69.34%
 - heuristic_vit: 69.34%
 - heuristic_beam: 74.34%


## 7. Zero-Shot Out-of-Distribution (GAPS) & JSON Report Export


In [8]:
# -----------------------------------------------------------------------------
# Out-of-Distribution validation checks for GAPS dataset (MIDI/MusicXML parse & score)
# -----------------------------------------------------------------------------
import xml.etree.ElementTree as ET
import mido

def parse_musicxml_guitar(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    notes = []
    for part in root.findall(".//part"):
        for measure in part.findall(".//measure"):
            for note_elem in measure.findall(".//note"):
                if note_elem.find("rest") is not None:
                    continue
                pitch_elem = note_elem.find("pitch")
                if pitch_elem is None:
                    continue
                step = pitch_elem.find("step").text
                octave = int(pitch_elem.find("octave").text)
                alter_elem = pitch_elem.find("alter")
                alter = int(float(alter_elem.text)) if alter_elem is not None else 0
                
                pc_map = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}
                midi = 12 * (octave + 1) + pc_map[step] + alter
                
                fret = None
                string = None
                tech = note_elem.find(".//technical")
                if tech is not None:
                    fret_elem = tech.find("fret")
                    string_elem = tech.find("string")
                    if fret_elem is not None:
                        fret = int(fret_elem.text)
                    if string_elem is not None:
                        string = 6 - int(string_elem.text)
                
                if fret is not None and string is not None:
                    notes.append({
                        "midi": midi,
                        "string": string,
                        "fret": fret
                    })
    return notes

def load_midi_notes(midi_path):
    mid = mido.MidiFile(midi_path)
    active_notes = {}
    notes = []
    current_time = 0.0
    for msg in mid:
        current_time += msg.time
        if msg.type == 'note_on' and msg.velocity > 0:
            active_notes[(msg.channel, msg.note)] = current_time
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            key = (msg.channel, msg.note)
            if key in active_notes:
                start_time = active_notes.pop(key)
                duration = current_time - start_time
                notes.append({
                    "start": start_time,
                    "duration": duration,
                    "midi": msg.note,
                })
    notes.sort(key=lambda x: x["start"])
    return notes

gaps_results = {"gaps_checked": False, "gaps_status": "skipped"}
if RUN_GAPS and GAPS_DIR.exists():
    print("Executing GAPS zero-shot validation (Mode B decoder-only OOD)...")
    xml_dir = GAPS_DIR / "musicxml"
    midi_dir = GAPS_DIR / "midi"
    
    xml_files = sorted(list(xml_dir.glob("*.xml")))
    eval_xmls = xml_files[:MAX_EVAL_TRACKS]
    
    gaps_mode_b_results = defaultdict(list)
    
    for xml_path in eval_xmls:
        stem = xml_path.stem
        midi_path = midi_dir / f"{stem}.mid"
        if not midi_path.exists():
            midi_path = next(midi_dir.glob(f"{stem}*.mid"), None)
        if not midi_path or not midi_path.exists():
            continue
            
        try:
            xml_notes = parse_musicxml_guitar(xml_path)
            midi_notes = load_midi_notes(midi_path)
            
            xml_by_pitch = defaultdict(list)
            for xn in xml_notes:
                xml_by_pitch[xn["midi"]].append(xn)
            midi_by_pitch = defaultdict(list)
            for mn in midi_notes:
                midi_by_pitch[mn["midi"]].append(mn)
                
            notes = []
            for pitch, m_list in midi_by_pitch.items():
                if not (40 <= pitch <= 88):
                    continue
                x_list = xml_by_pitch.get(pitch, [])
                for mn, xn in zip(m_list, x_list):
                    notes.append({
                        "start": mn["start"],
                        "duration": mn["duration"],
                        "midi": pitch,
                        "true_string": xn["string"],
                        "true_fret": xn["fret"]
                    })
            notes.sort(key=lambda x: x["start"])
            for idx, n in enumerate(notes):
                n["note_idx"] = idx
                
            if len(notes) < 4:
                continue
                
            oracle_alignment = {idx: gn for idx, gn in enumerate(notes)}
            
            dec_base = fb.assign_combined_all_tuned(copy.deepcopy(notes))
            gaps_mode_b_results["baseline"].append(score_exact_tab(dec_base, oracle_alignment))
            
            dec_down = repair_notebook_tab_assignments(dec_base)
            gaps_mode_b_results["downstream"].append(score_exact_tab(dec_down, oracle_alignment))
            
            rep_heur = filter_phantom_notes(copy.deepcopy(notes), threshold=0.40)
            dec_heur_vit = fb.assign_combined_all_tuned(rep_heur)
            gaps_mode_b_results["heuristic_vit"].append(score_exact_tab(dec_heur_vit, oracle_alignment))
            
            try:
                dec_heur_beam = fb.assign_prox_viterbi_transformer(rep_heur)
                gaps_mode_b_results["heuristic_beam"].append(score_exact_tab(dec_heur_beam, oracle_alignment))
            except Exception:
                gaps_mode_b_results["heuristic_beam"].append(np.nan)
        except Exception as e:
            print(f"Failed to score GAPS track {stem}: {e}")
            
    gaps_mean_results = {}
    for k, v in gaps_mode_b_results.items():
        valid_vals = [x for x in v if not pd.isna(x)]
        gaps_mean_results[k] = np.mean(valid_vals) if valid_vals else 0.0
        
    gaps_results = {
        "gaps_checked": True,
        "gaps_status": "success",
        "protocol_details": "OOD Oracle Assignment (decoder-only subset testing transcription generalization on non-studio audio score-aligned notes, not full end-to-end basic pitch transcription repair)",
        "mode_b_accuracies": gaps_mean_results,
        "n_tracks": len(eval_xmls)
    }
    print("GAPS zero-shot validation completed.")

# Check if precision gate was met in folds
gate_status = {}
for group, info in folds_data.items():
    gate_status[group] = info["gates"]

# Compute aggregated classification metrics per fold
mean_classification_metrics = defaultdict(dict)
for metric_name in ["precision", "recall", "f1"]:
    for model_name in ["xgb_v1", "xgb_v2", "rf_v1", "rf_v2", "mlp_v1", "mlp_v2", "seq_v1", "seq_v2"]:
        vals = [info["val_metrics"][model_name][metric_name] for info in folds_data.values() if "val_metrics" in info]
        mean_classification_metrics[model_name][metric_name] = float(np.mean(vals)) if vals else 0.0

# Compute coverage indicators
guitarset_total_stems = len(recording_datasets)
guitarset_evaluated_stems = eval_count
coverage_pct = (guitarset_evaluated_stems / guitarset_total_stems) * 100.0 if guitarset_total_stems > 0 else 0.0

report_out = {
    "device": str(DEVICE),
    "timestamp": time.time(),
    "max_eval_tracks": MAX_EVAL_TRACKS,
    "guitarset_coverage": {
        "total_stems": guitarset_total_stems,
        "evaluated_stems": guitarset_evaluated_stems,
        "coverage_percent": coverage_pct,
        "note_count_evaluated": sum(len(pd.read_csv(bp_path)) for stem, (jams_path, bp_path) in list(recording_datasets.items())[:eval_count])
    },
    "mean_classification_metrics_lopo": dict(mean_classification_metrics),
    "mode_a_accuracies": final_accuracies_a,
    "mode_b_accuracies": final_accuracies_b,
    "gaps_evaluation": gaps_results,
    "precision_gate_statuses": gate_status,
    "precision_gate_met": any(any(g.values()) for g in gate_status.values()),
    "mode_a_playability": {
        "collisions": {k: float(np.mean(v)) for k, v in mode_a_collisions.items()},
        "stretches": {k: float(np.mean(v)) for k, v in mode_a_stretches.items()}
    },
    "mode_a_latencies": {k: float(np.mean(v)) for k, v in mode_a_latencies.items()}
}

out_dir = repo_root / "outputs" / "fretwork_generalization"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "unified_report.json", "w") as f:
    json.dump(report_out, f, indent=4)

print("Unified report exported to outputs/fretwork_generalization/unified_report.json")


Executing GAPS zero-shot validation (Mode B decoder-only OOD)...
Failed to score GAPS track 189_qk1wc: ' '
Failed to score GAPS track 265_yw1wc: ' '
Failed to score GAPS track 305_Cc1wc: ' '
Failed to score GAPS track 357_D41wc: ' '
GAPS zero-shot validation completed.
Unified report exported to outputs/fretwork_generalization/unified_report.json
